# Suitability Analysis

Characterize the development potential and socioeconomic profile within 1–2 miles of Diridon Station to support transit-oriented development planning and opportunity analysis.

In [27]:
# --------------------------
# LIBRARIES
# --------------------------
# file path
import os
from pathlib import Path

# data management
import pandas as pd
import geopandas as gpd
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# modeling
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

# visualizing
import matplotlib.pyplot as plt

# settings
pd.set_option('display.max_columns', None)  # Show all columns when printing the DataFrame


In [33]:
# -----------------------------------------------------
# Step 1: Load Data
# -----------------------------------------------------
PROCESSED_DIR = Path("../data/processed")
PARCELS_FOR_ML = PROCESSED_DIR / "parcels_for_ml.parquet"
OUTPUT_DIR = Path("../output/suitability_model/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

parcels_model = gpd.read_parquet(PARCELS_FOR_ML)
parcels_model.head()



,OBJECTID_left,PARCELID,INTID_left,APN,LOTNUM,PARCELTYPE,FEATURECLA,PLANCRT,PLANMOD,LASTUPDATE_left,NOTES_left,COVERED,SHAPE_Leng_left,SHAPE_Area_left,CREATIONDA,geometry,OBJECTID_right,FACILITYID,INTID_right,ZONING,ZONINGABBR,REZONINGFI,PDUSE,PDDENSITY,DEVELOPEDA,APPROVALDA,COLORCODE,LASTUPDATE_right,NOTES_right,SHAPE_Leng_right,SHAPE_Area_right,overlap_area,zoning,zoning_class,zoning_planned,GEOID,public_transit_pct,walked_pct,drove_pct,pct_renters,vacancy_rate,median_rent,median_income,pct_white,pct_black,pct_asian,pct_latino,pct_college_plus,in_taz,urban_zone_flag,dist_to_station_miles,parcel_area
0,4378,1000,1000,09759046,46,Tax,Parcel,T-8284,None,2005-10-25,None,None,149.517344,1163.273287,1900-01-01,"POLYGON ((-121.92153 37.40344, -121.92148 37.4...",1396.0,1396,1396.0,A(PD),A(PD),88034,Res,25.6,Yes,None,17,2022-04-22,None,5039.839521,1.155021e+06,1163.273287,Other,Special Purpose,True,06085505006,3.273626,4.657196,60.370599,87.253849,7.577763,3210.0,205610.0,20.237282,0.887675,76.237624,4.933424,89.557753,0,0,6.621561,1.099912e-08
1,173262,1000000031,1000000031,30326084,6,Tax,Parcel,TR9494,None,2006-01-12,None,None,188.703136,2037.538177,1900-01-01,"POLYGON ((-121.96725 37.32211, -121.96704 37.3...",288.0,288,288.0,A(PD),A(PD),02028,Res,13,Yes,None,17,2022-04-22,None,737.405158,2.014218e+04,2037.538177,Other,Special Purpose,True,06085506301,2.352193,5.181182,68.817546,70.102136,10.362047,2461.0,121667.0,44.893158,4.763772,29.876495,19.407959,46.546175,1,0,4.607317,1.924470e-08
2,69247,1000000032,1000000032,30326083,5,Tax,Parcel,TR9494,None,2006-01-12,None,None,187.780061,2020.818784,1900-01-01,"POLYGON ((-121.96725 37.32202, -121.96704 37.3...",288.0,288,288.0,A(PD),A(PD),02028,Res,13,Yes,None,17,2022-04-22,None,737.405158,2.014218e+04,2020.818784,Other,Special Purpose,True,06085506301,2.352193,5.181182,68.817546,70.102136,10.362047,2461.0,121667.0,44.893158,4.763772,29.876495,19.407959,46.546175,1,0,4.608391,1.908676e-08
3,147109,1000000033,1000000033,30326082,4,Tax,Parcel,TR9494,None,2006-01-12,None,None,187.209294,1994.378436,1900-01-01,"POLYGON ((-121.96691 37.32205, -121.96689 37.3...",288.0,288,288.0,A(PD),A(PD),02028,Res,13,Yes,None,17,2022-04-22,None,737.405158,2.014218e+04,1994.378436,Other,Special Purpose,True,06085506301,2.352193,5.181182,68.817546,70.102136,10.362047,2461.0,121667.0,44.893158,4.763772,29.876495,19.407959,46.546175,1,0,4.590963,1.883703e-08
4,147110,1000000034,1000000034,30326081,3,Tax,Parcel,TR9494,None,2006-01-12,None,None,212.838294,2261.152730,1900-01-01,"POLYGON ((-121.9666 37.32198, -121.9666 37.321...",288.0,288,288.0,A(PD),A(PD),02028,Res,13,Yes,None,17,2022-04-22,None,737.405158,2.014218e+04,2261.152730,Other,Special Purpose,True,06085506301,2.352193,5.181182,68.817546,70.102136,10.362047,2461.0,121667.0,44.893158,4.763772,29.876495,19.407959,46.546175,1,0,4.578673,2.135674e-08


In [34]:
# =============================================================
# Step 2: Separate into features and labels 
# =============================================================

feature_cols = [
    'dist_to_station_miles',
    'parcel_area',
    'in_taz',
    'public_transit_pct', 
    'walked_pct', 
    'drove_pct', 
    'pct_renters',
    'vacancy_rate',
    'median_income',
    'median_rent',
    'pct_white',
    'pct_black',
    'pct_asian',
    'pct_latino',
    'pct_college_plus'
] 

feature_cols

X = parcels_model[feature_cols]
y = parcels_model["urban_zone_flag"]

In [35]:
# =============================================================
# Step 3: Train/Test Split
# =============================================================

X = parcels_model[feature_cols]
y = parcels_model["urban_zone_flag"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Scale continuous variables
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [36]:
# =============================================================
# Step 4: Train ML Models
# =============================================================

print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=350,
    max_depth=18,
    min_samples_leaf=8,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

print("Training XGBoost...")
xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)
xgb.fit(X_train, y_train)

print("Training Voting Ensemble...")
ensemble = VotingClassifier(
    estimators=[("rf", rf), ("xgb", xgb)],
    voting="soft",
    weights=[1, 2]
)
ensemble.fit(X_train, y_train)

Training Random Forest...
Training XGBoost...
Training Voting Ensemble...


,estimators,"[('rf', ...), ('xgb', ...)]"
,voting,'soft'
,weights,"[1, 2]"
,n_jobs,None
,flatten_transform,True
,verbose,False
,n_estimators,350
,criterion,'gini'
,max_depth,18
,min_samples_split,2
,min_samples_leaf,8


In [25]:

# =============================================================
# Step 5: Evaluate Models
# =============================================================

def evaluate(model, X_test, y_test, name):
    print(f"\n---- {name} PERFORMANCE ----")
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, probs)
    print("ROC AUC:", round(auc, 3))
    print(classification_report(y_test, preds))

evaluate(rf, X_test, y_test, "Random Forest")
evaluate(xgb, X_test, y_test, "XGBoost")
evaluate(ensemble, X_test, y_test, "Ensemble")



---- Random Forest PERFORMANCE ----
ROC AUC: 0.986
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     47893
           1       0.81      0.48      0.60      1001

    accuracy                           0.99     48894
   macro avg       0.90      0.74      0.80     48894
weighted avg       0.99      0.99      0.99     48894


---- XGBoost PERFORMANCE ----
ROC AUC: 0.986
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     47893
           1       0.78      0.52      0.62      1001

    accuracy                           0.99     48894
   macro avg       0.88      0.76      0.81     48894
weighted avg       0.99      0.99      0.99     48894


---- Ensemble PERFORMANCE ----
ROC AUC: 0.987
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     47893
           1       0.80      0.50      0.62      1001

    accuracy                         

In [41]:
# =============================================================
# Step 6: Feature Importance Plots
# =============================================================

print("Plotting feature importances...")

importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances.sort_values().plot.barh(figsize=(10, 12))

plt.title("Random Forest Feature Importances")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rf_feature_importance.png", dpi=160)
plt.close()

Plotting feature importances...


In [39]:
importances

dist_to_station_miles    0.377664
parcel_area              0.299090
in_taz                   0.021155
public_transit_pct       0.017548
walked_pct               0.019141
drove_pct                0.018511
pct_renters              0.052802
vacancy_rate             0.026469
median_income            0.023229
median_rent              0.026223
pct_white                0.022667
pct_black                0.030991
pct_asian                0.023009
pct_latino               0.021589
pct_college_plus         0.019912
dtype: float64

In [44]:
# =============================================================
# Step 7: Predict Suitability for All Parcels
# =============================================================

print("Predicting suitability probabilities...")
parcels_model["suitability_score"] = ensemble.predict_proba(
    scaler.transform(parcels_model[feature_cols].fillna(0))
)[:, 1]

# Identify parcels the model thinks *should* be urban but aren't
parcels_model["predicted_urban_flag"] = (parcels_model["suitability_score"] > 0.5).astype(int)

parcels_model["potential_rezoning_candidate"] = np.where(
    (parcels_model["predicted_urban_flag"] == 1) &
    (parcels_model["urban_zone_flag"] == 0),
    1, 0
)

# Save outputs
parcels_model.to_parquet(OUTPUT_DIR / "parcels_with_suitability.parquet")

print("Saving GeoJSON of potentially mis-zoned parcels...")
mis_zoned = parcels_model[parcels_model["potential_rezoning_candidate"] == 0]
mis_zoned.to_file(OUTPUT_DIR / "mis_zoned_opportunity.geojson", driver="GeoJSON")

print("\n✓ ML Suitability Modeling complete!")
print(f"Outputs saved to: {OUTPUT_DIR}")

Predicting suitability probabilities...


/opt/anaconda3/envs/housing_project/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Saving GeoJSON of potentially mis-zoned parcels...

✓ ML Suitability Modeling complete!
Outputs saved to: ../output/suitability_model


In [ ]:
parcels_model


,OBJECTID_left,PARCELID,INTID_left,APN,LOTNUM,PARCELTYPE,FEATURECLA,PLANCRT,PLANMOD,LASTUPDATE_left,NOTES_left,COVERED,SHAPE_Leng_left,SHAPE_Area_left,CREATIONDA,geometry,OBJECTID_right,FACILITYID,INTID_right,ZONING,ZONINGABBR,REZONINGFI,PDUSE,PDDENSITY,DEVELOPEDA,APPROVALDA,COLORCODE,LASTUPDATE_right,NOTES_right,SHAPE_Leng_right,SHAPE_Area_right,overlap_area,zoning,zoning_class,zoning_planned,GEOID,public_transit_pct,walked_pct,drove_pct,pct_renters,vacancy_rate,median_rent,median_income,pct_white,pct_black,pct_asian,pct_latino,pct_college_plus,in_taz,urban_zone_flag,dist_to_station_miles,parcel_area,suitability_score,predicted_urban_flag,potential_rezoning_candidate
0,4378,1000,1000,09759046,46,Tax,Parcel,T-8284,None,2005-10-25,None,None,149.517344,1163.273287,1900-01-01,"POLYGON ((-121.92153 37.40344, -121.92148 37.4...",1396.0,1396,1396.0,A(PD),A(PD),88034,Res,25.6,Yes,None,17,2022-04-22,None,5039.839521,1.155021e+06,1163.273287,Other,Special Purpose,True,06085505006,3.273626,4.657196,60.370599,87.253849,7.577763,3210.0,205610.0,20.237282,0.887675,76.237624,4.933424,89.557753,0,0,6.621561,1.099912e-08,0.068001,0,0
1,173262,1000000031,1000000031,30326084,6,Tax,Parcel,TR9494,None,2006-01-12,None,None,188.703136,2037.538177,1900-01-01,"POLYGON ((-121.96725 37.32211, -121.96704 37.3...",288.0,288,288.0,A(PD),A(PD),02028,Res,13,Yes,None,17,2022-04-22,None,737.405158,2.014218e+04,2037.538177,Other,Special Purpose,True,06085506301,2.352193,5.181182,68.817546,70.102136,10.362047,2461.0,121667.0,44.893158,4.763772,29.876495,19.407959,46.546175,1,0,4.607317,1.924470e-08,0.089043,0,0
2,69247,1000000032,1000000032,30326083,5,Tax,Parcel,TR9494,None,2006-01-12,None,None,187.780061,2020.818784,1900-01-01,"POLYGON ((-121.96725 37.32202, -121.96704 37.3...",288.0,288,288.0,A(PD),A(PD),02028,Res,13,Yes,None,17,2022-04-22,None,737.405158,2.014218e+04,2020.818784,Other,Special Purpose,True,06085506301,2.352193,5.181182,68.817546,70.102136,10.362047,2461.0,121667.0,44.893158,4.763772,29.876495,19.407959,46.546175,1,0,4.608391,1.908676e-08,0.089043,0,0
3,147109,1000000033,1000000033,30326082,4,Tax,Parcel,TR9494,None,2006-01-12,None,None,187.209294,1994.378436,1900-01-01,"POLYGON ((-121.96691 37.32205, -121.96689 37.3...",288.0,288,288.0,A(PD),A(PD),02028,Res,13,Yes,None,17,2022-04-22,None,737.405158,2.014218e+04,1994.378436,Other,Special Purpose,True,06085506301,2.352193,5.181182,68.817546,70.102136,10.362047,2461.0,121667.0,44.893158,4.763772,29.876495,19.407959,46.546175,1,0,4.590963,1.883703e-08,0.089043,0,0
4,147110,1000000034,1000000034,30326081,3,Tax,Parcel,TR9494,None,2006-01-12,None,None,212.838294,2261.152730,1900-01-01,"POLYGON ((-121.9666 37.32198, -121.9666 37.321...",288.0,288,288.0,A(PD),A(PD),02028,Res,13,Yes,None,17,2022-04-22,None,737.405158,2.014218e+04,2261.152730,Other,Special Purpose,True,06085506301,2.352193,5.181182,68.817546,70.102136,10.362047,2461.0,121667.0,44.893158,4.763772,29.876495,19.407959,46.546175,1,0,4.578673,2.135674e-08,0.089043,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224903,4370,998,998,09759044,44,Tax,Parcel,T-8284,None,2005-10-25,None,None,165.310559,1643.340351,1900-01-01,"POLYGON ((-121.92179 37.40337, -121.92178 37.4...",1396.0,1396,1396.0,A(PD),A(PD),88034,Res,25.6,Yes,None,17,2022-04-22,None,5039.839521,1.155021e+06,1643.340351,Other,Special Purpose,True,06085505006,3.273626,4.657196,60.370599,87.253849,7.577763,3210.0,205610.0,20.237282,0.887675,76.237624,4.933424,89.557753,0,0,6.617616,1.553829e-08,0.068001,0,0
224904,4371,999,999,09759045,45,Tax,Parcel,T-8284,None,2005-10-25,None,None,168.419025,1661.036864,1900-01-01,"POLYGON ((-121.92172 37.40341, -121.9216 37.40...",1396.0,1396,1396.0,A(PD),A(PD),88034,Res,25.6,Yes,None,17,2022-04-22,None,5039.839521,1.155021e+06,1661.036864,Other,Special Purpose,True,0608550

In [46]:
mis_zoned

,OBJECTID_left,PARCELID,INTID_left,APN,LOTNUM,PARCELTYPE,FEATURECLA,PLANCRT,PLANMOD,LASTUPDATE_left,NOTES_left,COVERED,SHAPE_Leng_left,SHAPE_Area_left,CREATIONDA,geometry,OBJECTID_right,FACILITYID,INTID_right,ZONING,ZONINGABBR,REZONINGFI,PDUSE,PDDENSITY,DEVELOPEDA,APPROVALDA,COLORCODE,LASTUPDATE_right,NOTES_right,SHAPE_Leng_right,SHAPE_Area_right,overlap_area,zoning,zoning_class,zoning_planned,GEOID,public_transit_pct,walked_pct,drove_pct,pct_renters,vacancy_rate,median_rent,median_income,pct_white,pct_black,pct_asian,pct_latino,pct_college_plus,in_taz,urban_zone_flag,dist_to_station_miles,parcel_area,suitability_score,predicted_urban_flag,potential_rezoning_candidate
0,4378,1000,1000,09759046,46,Tax,Parcel,T-8284,None,2005-10-25,None,None,149.517344,1163.273287,1900-01-01,"POLYGON ((-121.92153 37.40344, -121.92148 37.4...",1396.0,1396,1396.0,A(PD),A(PD),88034,Res,25.6,Yes,None,17,2022-04-22,None,5039.839521,1.155021e+06,1163.273287,Other,Special Purpose,True,06085505006,3.273626,4.657196,60.370599,87.253849,7.577763,3210.0,205610.0,20.237282,0.887675,76.237624,4.933424,89.557753,0,0,6.621561,1.099912e-08,0.068001,0,0
1,173262,1000000031,1000000031,30326084,6,Tax,Parcel,TR9494,None,2006-01-12,None,None,188.703136,2037.538177,1900-01-01,"POLYGON ((-121.96725 37.32211, -121.96704 37.3...",288.0,288,288.0,A(PD),A(PD),02028,Res,13,Yes,None,17,2022-04-22,None,737.405158,2.014218e+04,2037.538177,Other,Special Purpose,True,06085506301,2.352193,5.181182,68.817546,70.102136,10.362047,2461.0,121667.0,44.893158,4.763772,29.876495,19.407959,46.546175,1,0,4.607317,1.924470e-08,0.089043,0,0
2,69247,1000000032,1000000032,30326083,5,Tax,Parcel,TR9494,None,2006-01-12,None,None,187.780061,2020.818784,1900-01-01,"POLYGON ((-121.96725 37.32202, -121.96704 37.3...",288.0,288,288.0,A(PD),A(PD),02028,Res,13,Yes,None,17,2022-04-22,None,737.405158,2.014218e+04,2020.818784,Other,Special Purpose,True,06085506301,2.352193,5.181182,68.817546,70.102136,10.362047,2461.0,121667.0,44.893158,4.763772,29.876495,19.407959,46.546175,1,0,4.608391,1.908676e-08,0.089043,0,0
3,147109,1000000033,1000000033,30326082,4,Tax,Parcel,TR9494,None,2006-01-12,None,None,187.209294,1994.378436,1900-01-01,"POLYGON ((-121.96691 37.32205, -121.96689 37.3...",288.0,288,288.0,A(PD),A(PD),02028,Res,13,Yes,None,17,2022-04-22,None,737.405158,2.014218e+04,1994.378436,Other,Special Purpose,True,06085506301,2.352193,5.181182,68.817546,70.102136,10.362047,2461.0,121667.0,44.893158,4.763772,29.876495,19.407959,46.546175,1,0,4.590963,1.883703e-08,0.089043,0,0
4,147110,1000000034,1000000034,30326081,3,Tax,Parcel,TR9494,None,2006-01-12,None,None,212.838294,2261.152730,1900-01-01,"POLYGON ((-121.9666 37.32198, -121.9666 37.321...",288.0,288,288.0,A(PD),A(PD),02028,Res,13,Yes,None,17,2022-04-22,None,737.405158,2.014218e+04,2261.152730,Other,Special Purpose,True,06085506301,2.352193,5.181182,68.817546,70.102136,10.362047,2461.0,121667.0,44.893158,4.763772,29.876495,19.407959,46.546175,1,0,4.578673,2.135674e-08,0.089043,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224903,4370,998,998,09759044,44,Tax,Parcel,T-8284,None,2005-10-25,None,None,165.310559,1643.340351,1900-01-01,"POLYGON ((-121.92179 37.40337, -121.92178 37.4...",1396.0,1396,1396.0,A(PD),A(PD),88034,Res,25.6,Yes,None,17,2022-04-22,None,5039.839521,1.155021e+06,1643.340351,Other,Special Purpose,True,06085505006,3.273626,4.657196,60.370599,87.253849,7.577763,3210.0,205610.0,20.237282,0.887675,76.237624,4.933424,89.557753,0,0,6.617616,1.553829e-08,0.068001,0,0
224904,4371,999,999,09759045,45,Tax,Parcel,T-8284,None,2005-10-25,None,None,168.419025,1661.036864,1900-01-01,"POLYGON ((-121.92172 37.40341, -121.9216 37.40...",1396.0,1396,1396.0,A(PD),A(PD),88034,Res,25.6,Yes,None,17,2022-04-22,None,5039.839521,1.155021e+06,1661.036864,Other,Special Purpose,True,0608550

In [47]:
parcels_model.shape  

(195574, 55)